In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [ ]:
# 1. Carregar os dados
df_dados = pd.read_excel("Base Leite 2023-2024 Formatada.xlsx")
df_dados

In [ ]:
df_dados.groupby('sistema')['id_fazenda'].nunique()

In [ ]:
df_dados.info()

In [ ]:
df_dados = df_dados.drop_duplicates()

In [ ]:
df_dados["gastos_totais_na_atividade_renda_bruta_da_atividade_percentual"] = df_dados["59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual"] + df_dados["60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual"] + df_dados["61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual"]

In [ ]:
df_dados_2023 = df_dados[df_dados["ano"] == 2023]
df_dados_2024 = df_dados[df_dados["ano"] == 2024]

In [ ]:
# 2. Definir variáveis de entrada (features) e a variável alvo (target)
features_standard = [
    # "gastos_totais_na_atividade_renda_bruta_da_atividade_percentual",
    "61_gasto_com_concentrado_na_atividade_renda_bruta_da_atividade_percentual",
    "60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual",
    "27_producao_mao_de_obra_total_litros_trabalhador_dia",
    "18_vacas_em_lactacao_total_de_animais_percentual",
    "26_producao_total_de_vacas_litros_dia"
]

features_minmax = [
    # "gastos_totais_na_atividade_renda_bruta_da_atividade_percentual",
    "59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual",
    "21_vacas_em_lactacao_mao_de_obra_total_animais_trabalhador_dia"
]

features_robust = [
    # "gastos_totais_na_atividade_renda_bruta_da_atividade_percentual",
    "35_relacao_de_troca_leite_concentrado_kg_l",
    "33_preco_medio_do_leite_r_litro",
    "20_vacas_em_lactacao_area_destinada_a_atividade_considerando_reserva_animais_hectare"
    
]

features_cat = ["id_fazenda", "sistema"]

target = "52_custo_total_do_leite_r_litro"

todas_features = features_standard + features_minmax + features_robust + features_cat
data = df_dados[todas_features + ['ano', target]].dropna()

train_data = data[data['ano'] == 2023]
test_data = data[data['ano'] == 2024]

# O 'ano' NÃO entra no X_train nem no X_test, pois ele é constante dentro de cada grupo
X_train = train_data[todas_features]
y_train = train_data[target]

X_test = test_data[todas_features]
y_test = test_data[target]

# 4. Criar o pré-processador
# Variáveis numéricas: são padronizadas (StandardScaler)
# Variáveis categóricas: são transformadas em colunas binárias (OneHotEncoder)
preprocessor = ColumnTransformer(
    transformers=[
        ('std_scaler', StandardScaler(), features_standard),
        ('minmax_scaler', MinMaxScaler(), features_minmax),
        ('robust_scaler', RobustScaler(), features_robust),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat)
    ]
)

# 5. Dicionário com os Modelos de Machine Learning que serão testados
models = {
    "Regressão Linear": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, random_state=42),
    "Rede Neural (Rasa - 64 neurônios)": MLPRegressor(
        hidden_layer_sizes=(64,), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Rede Neural (Média - 64x32)": MLPRegressor(
        hidden_layer_sizes=(64, 32), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Rede Neural (Profunda - 128x64x32)": MLPRegressor(
        hidden_layer_sizes=(128, 64, 32), solver='lbfgs', max_iter=2000, random_state=42
    ),
    "Regressão Ridge (Penalizada)": Ridge(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "Support Vector Regression (SVR)": SVR(kernel='rbf', C=1.0, epsilon=0.1),
    "LightGBM (HistGradientBoosting)": HistGradientBoostingRegressor(random_state=42)
}

# Lista para salvar os resultados
results = []

# 7. Treinamento e Avaliação
for name, model in models.items():
    # Cria o pipeline que vai rodar o pré-processamento antes de entregar ao modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Treina o modelo
    pipeline.fit(X_train, y_train)
    
    # Previsões nos dados de teste
    y_pred = pipeline.predict(X_test)
    
    # Cálculos de Erros
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Modelo": name,
        "MAE (R$/L)": round(mae, 4),
        "RMSE (R$/L)": round(rmse, 4),
        "R²": round(r2, 4)
    })

# 8. Mostrar a tabela com os resultados finais
results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

In [ ]:
df_dados["custo_concentrado_r_ano"] = df_dados["9_consumo_de_concentrado_anual_kg_ano"] * df_dados["34_preco_medio_do_concentrado_r_kg"]
df_dados["custo_volumoso_r_L"] = (df_dados["31_renda_bruta_da_atividade_leiteira_r_ano"] * (df_dados["60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual"]/100)) / df_dados["22_producao_anual_de_leite_litros_ano"]
df_dados["custo_maodeobra_r_ano"] = (df_dados["59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual"]/100) * df_dados["31_renda_bruta_da_atividade_leiteira_r_ano"]
df_dados["custo_maodeobra_r_L"] = (df_dados["31_renda_bruta_da_atividade_leiteira_r_ano"] * (df_dados["59_gasto_com_mao_de_obra_contratada_na_atividade_renda_bruta_da_atividade_percentual"]/100)) / df_dados["22_producao_anual_de_leite_litros_ano"]

In [ ]:
print(206426/df_dados["custo_concentrado_r_ano"].mean(), 0.1824/df_dados["custo_volumoso_r_L"].mean(), 48091.1/df_dados["custo_maodeobra_r_ano"].mean())

In [ ]:
df_dados["22_producao_anual_de_leite_litros_ano"]

In [ ]:
df_dados["31_renda_bruta_da_atividade_leiteira_r_ano"]

In [ ]:
df_dados["60_gasto_com_volumoso_na_atividade_renda_bruta_da_atividade_percentual"]

In [ ]:
df_dados["custo_maodeobra_r_L"]

In [ ]:
features = [
    "10_mao_de_obra_total_trabalhador_dia"
]

features_cat = ["id_fazenda", "sistema"]

target = "custo_maodeobra_r_ano"

todas_features = features + features_cat

data = df_dados[todas_features + ['ano', target]].dropna()

train_data = data[data['ano'] == 2023]
test_data = data[data['ano'] == 2024]

X_train = train_data[todas_features]
y_train = train_data[target]

X_test = test_data[todas_features]
y_test = test_data[target]

preprocessor = ColumnTransformer(
    transformers=[
        ('std_scaler', StandardScaler(), features),
        # ('minmax_scaler', MinMaxScaler(), features),
        # ('robust_scaler', RobustScaler(), features),
        ('cat_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat)
    ]
)

models = {
    "Regressão Linear": LinearRegression(),
    "Regressão Ridge (Penalizada)": Ridge(alpha=1.0)
}

results = []
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Modelo": name,
        "MAE (R$/ano)": round(mae, 4),
        "RMSE (R$/ano)": round(rmse, 4),
        "R²": round(r2, 4)
    })

results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

In [ ]:
# 4. Plot the results
# Plot the actual data points (using the test set for clarity)
plt.scatter(X_test, y_test, color='black', label='Actual data')

# Plot the regression line
plt.plot(X_test, y_pred, color='blue', linewidth=2, label='Regression line')

# Add labels and a legend
plt.title('Linear Regression Results')
plt.xlabel('Independent Variable (X)')
plt.ylabel('Dependent Variable (Y)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--') # Perfect prediction line
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs. Predicted Values')
plt.show()